# pi0.5: does the placement rule hold on a second model?
| PART | Runs | Approx. time | Inputs besides vla-assure |
|---|---|---|---|
| 1 | condition check (4 x 50 episodes at most) + 400 reference episodes | ~6-8 h | none |
| 2 | W4 and W8 on the action expert (140 test + 80 repair slots each) | ~6-8 h | output of PART 1 |
| 3 | W4 on language model + expert, and the second checkpoint | ~6-8 h | output of PART 2 |
| 4 | optional: 5 denoising steps, 5 actions per call | ~6-8 h | output of PART 3 |

**Setup (every part):** GPU T4 x2, Internet On, secret `HF_TOKEN` switched on, Add Input: dataset **vla-assure** (latest)
and, from PART 2 on, **Notebook Output** of the previous version of THIS notebook. Set `PART` in cell 2a, then
**Save Version -> Save & Run All (Commit)**. **Send back:** `pi05_partN.zip` from the Output tab.
If a part stops at ~11 h with "OutOfTime", commit the SAME part again with that version's output attached.

## 1. Setup

In [ ]:
# 1a. GPUs, time guard, work folder; resume from a previous version's output if one is attached as input.
import os, sys, time, json, glob, shutil, pathlib, subprocess
T_START = time.time()
TIME_BUDGET_H = 11.0          # stop cleanly before Kaggle's 12 h limit so the output is saved
def hours_left():
    return TIME_BUDGET_H - (time.time() - T_START) / 3600

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
assert NGPU >= 1, "No GPU: right panel > Accelerator > GPU T4 x2"
print("GPUs:", NGPU, "| kernel Python", sys.version.split()[0])

WORK = pathlib.Path("/kaggle/working/vla-gonogo"); WORK.mkdir(parents=True, exist_ok=True)
# Kaggle mounts inputs at different depths (e.g. /kaggle/input/notebooks/<user>/<slug>/vla-gonogo)
prev = sorted({p for p in glob.glob("/kaggle/input/**/vla-gonogo", recursive=True) if pathlib.Path(p).is_dir()})
for p in prev:
    if pathlib.Path(p).resolve() != WORK.resolve():
        print("Resuming from previous output:", p)
        shutil.copytree(p, WORK, dirs_exist_ok=True, symlinks=False)
n_done = len(list(WORK.rglob("*.npz")))
print("Work folder:", WORK, "|", n_done, "episodes already on disk")
PERSISTENT = True   # /kaggle/working is saved as the version's output

In [ ]:
# 1b. Code: the newest vla-assure copy among the inputs (needs the dtype switch added for pi0.5)
import zipfile
CODE = pathlib.Path("/root/vla-assure")
def newest(root):
    root = pathlib.Path(root)
    a = root / "src/assure/adapters/lerobot_adapter.py"
    return a.exists() and "pcfg.dtype" in a.read_text() and (root / "scripts/resplit.py").exists()
if CODE.exists() and not newest(CODE):
    shutil.rmtree(CODE)
if not CODE.exists():
    trees = sorted({str(pathlib.Path(p).parents[3]) for p in
                    glob.glob("/kaggle/input/**/src/assure/adapters/lerobot_adapter.py", recursive=True)})
    trees = [t for t in trees if newest(t)]
    if not trees:
        raise FileNotFoundError("No vla-assure copy with the pi0.5 dtype switch: upload the newest vla-assure.zip "
                                "as a new dataset version and attach it")
    shutil.copytree(trees[0], CODE, symlinks=False)
for d in ("rollouts", "results"):
    (WORK / d).mkdir(exist_ok=True)
    tgt = CODE / d
    if tgt.is_symlink(): tgt.unlink()
    elif tgt.exists(): shutil.rmtree(tgt)
    tgt.symlink_to(WORK / d)
os.chdir(CODE); print("Code OK at", os.getcwd())

In [ ]:
# 1b'. Hugging Face token (pi0.5 uses Google's gated PaliGemma tokenizer)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF_TOKEN loaded (length", len(os.environ["HF_TOKEN"]), ")")
except Exception as e:
    raise RuntimeError("Add-ons -> Secrets: add HF_TOKEN and switch it on for this notebook") from e

In [ ]:
# 1c. Python 3.12 venv for LeRobot 0.6.1 (needs Python >= 3.12; Kaggle's kernel env is left untouched).
#     egl_probe / hf-egl-probe (via hf-libero -> robomimic) are C++ packages that fail to build with CMake 4;
#     LeRobot's LIBERO env never imports them, so tiny pure-Python stand-ins are installed first.
VENV = pathlib.Path("/root/py312"); PY = str(VENV / "bin/python")
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       env=dict(os.environ, CMAKE_POLICY_VERSION_MINIMUM="3.5"))
    if r.returncode != 0 or not check:
        print(r.stdout[-1500:], r.stderr[-3000:])
    if check:
        assert r.returncode == 0, f"failed: {cmd}"
    return r
if not pathlib.Path(PY).exists():
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv --seed --python 3.12 {VENV}")
stub = pathlib.Path("/root/_stubs")
(stub / "egl_probe/egl_probe").mkdir(parents=True, exist_ok=True); (stub / "hf_egl_probe").mkdir(parents=True, exist_ok=True)
(stub / "egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "egl_probe"\nversion = "1.0.2"\n[tool.setuptools]\npackages = ["egl_probe"]\n')
(stub / "egl_probe/egl_probe/__init__.py").write_text("def get_available_devices():\n    return [0]\n")
(stub / "hf_egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "hf-egl-probe"\nversion = "1.0.2"\n[tool.setuptools]\npy-modules = []\n')
sh(f"{PY} -m pip install -q {stub / 'egl_probe'} {stub / 'hf_egl_probe'}")
t = time.time()
sh(f'{PY} -m pip install -q "lerobot[pi,smolvla,libero]==0.6.1" pandas scipy tabulate pyyaml')
print(f"installed in {time.time() - t:.0f} s:",
      sh(f"{PY} -c \"import importlib.metadata as m, torch; print('lerobot', m.version('lerobot'), '| torch', "
         f"torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count())\"").stdout)

In [ ]:
# 1c'. Can this token read the gated tokenizer? (fails fast with a clear message instead of deep inside LeRobot)
r = subprocess.run([PY, "-c", "from transformers import AutoTokenizer; "
                    "AutoTokenizer.from_pretrained('google/paligemma-3b-pt-224'); print('tokenizer OK')"],
                   capture_output=True, text=True, env=dict(os.environ))
print((r.stdout + r.stderr)[-800:])
assert r.returncode == 0, ("token cannot read google/paligemma-3b-pt-224: open that page on huggingface.co with the "
                           "same account, click 'Agree and access', then rerun this cell")

In [ ]:
# 1d. Headless rendering. Try NVIDIA EGL first; if MuJoCo cannot render, install Mesa EGL (software) and retry;
#     last resort OSMesa. The chosen backend is used by every later step.
probe = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos=\"0 0 1\"/>"
         "<geom type=\"sphere\" size=\".1\"/></worldbody></mujoco>'); r = mujoco.Renderer(m, 64, 64); "
         "d = mujoco.MjData(m); mujoco.mj_forward(m, d); r.update_scene(d); print('render ok', r.render().shape)")
def try_gl(gl):
    r = subprocess.run([PY, "-c", probe], capture_output=True, text=True,
                       env=dict(os.environ, MUJOCO_GL=gl, PYOPENGL_PLATFORM=gl))
    return r.returncode == 0, (r.stdout + r.stderr)[-400:]
GL = None
for attempt in ("egl", "egl+mesa", "osmesa"):
    if attempt == "egl+mesa":
        sh("apt-get -qq update && apt-get -qq install -y libegl1 libegl-mesa0 libgl1 libgl1-mesa-dri >/dev/null", check=False)
    if attempt == "osmesa":
        sh("apt-get -qq install -y libosmesa6 >/dev/null", check=False)
    ok, log = try_gl(attempt.split("+")[0])
    print(attempt, "->", "OK" if ok else log)
    if ok:
        GL = attempt.split("+")[0]; break
assert GL, "no rendering backend works"
if attempt == "egl+mesa":
    print("NOTE: software EGL (Mesa). Rendering is CPU-bound; expect slower episodes than on Colab.")
os.environ["MUJOCO_GL"] = GL; os.environ["PYOPENGL_PLATFORM"] = GL
os.environ["MPLBACKEND"] = "Agg"   # the kernel's inline backend is not valid inside the venv's matplotlib
# LIBERO's first import asks interactive questions and downloads assets: answer once, non-interactively.
r = subprocess.run([PY, "-c", "import libero.libero as L; print('assets:', L.get_libero_path('assets'))"],
                   input="n\n", capture_output=True, text=True, env=dict(os.environ))
print(r.stdout[-600:], r.stderr[-600:])

In [ ]:
# 1e. Experiment configuration for this gate (written into the repo copy on this machine only)
import yaml, json
CFG = CODE / "configs"
exp = yaml.safe_load((CFG / "experiment.yaml").read_text())
models = yaml.safe_load((CFG / "models.yaml").read_text())

MODEL, SUITE = "pi05", "libero_spatial"
N_ACTION_STEPS = 10     # same throughput setting as SmolVLA
DTYPE = "bfloat16"      # float32 does not fit a T4 (pilot)
X = "model.paligemma_with_expert."
exp["quant_scope"] = {"include": [X + "gemma_expert.*"], "block_regex": r"gemma_expert\.model\.layers\.(\d+)$"}
exp["variants"]["w4_lm"] = {"kind": "quant", "bits": 4, "group_size": 128, "own_scope": True,
                            "include": [X + "paligemma.model.language_model.*", X + "gemma_expert.*"]}
exp["variants"]["w8_lm"] = dict(exp["variants"]["w4_lm"], bits=8)
models["models"][MODEL]["n_action_steps"] = N_ACTION_STEPS
def write_cfg(dtype):
    models["models"][MODEL]["dtype"] = dtype
    (CFG / "experiment.yaml").write_text(yaml.safe_dump(exp, sort_keys=False))
    (CFG / "models.yaml").write_text(yaml.safe_dump(models, sort_keys=False))
write_cfg(DTYPE)
print("checkpoint:", models["models"][MODEL]["checkpoint"], "| update:", models["models"][MODEL].get("update_checkpoint"))
print("taps:", models["models"][MODEL]["tap"])

def run(cmd, tail=3000):
    # Run a pipeline script, stream nothing, print the end of the log, raise on failure.
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    p = subprocess.run([PY] + cmd, capture_output=True, text=True, env=env, input="n\n")
    out = (p.stdout + p.stderr).replace("\r", "\n")
    out = "\n".join(l for l in out.splitlines() if "Running rollout" not in l and "it/s" not in l)
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


class OutOfTime(Exception):
    pass

def _clean_partial(since):
    # remove episode files that a killed shard may have left half-written
    import numpy as np
    for f in (CODE / "rollouts").rglob("*.npz"):
        if f.stat().st_mtime >= since - 120:
            try:
                np.load(f)["success"]
            except Exception:
                print("removing partial file", f); f.unlink()

def collect(args, tail=1500):
    # 01_collect.py sharded by task over the GPUs, then one merge pass (skips finished episodes) that writes the
    # complete run.json. Stops cleanly when TIME_BUDGET_H is reached; rerun (next version) to resume.
    if hours_left() < 0.25:
        raise OutOfTime("time budget reached: Save Version now; attach this output and run again to resume")
    args = list(args) + ([] if "--skip_existing" in args else ["--skip_existing"])
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    if NGPU > 1 and "--tasks" not in args:
        logs = [open(f"/root/shard{g}.log", "w") for g in range(NGPU)]
        t0 = time.time()
        procs = [subprocess.Popen([PY, "scripts/01_collect.py"] + args +
                                  ["--tasks", ",".join(str(t) for t in range(10) if t % NGPU == g)],
                                  # all shards on GPU 0 (the setup that works with robosuite's EGL checks); SmolVLA is small and the
                                  # simulator is CPU-bound, so two processes on one T4 still give ~2x
                                  env=dict(env, CUDA_VISIBLE_DEVICES="0"), stdout=logs[g], stderr=subprocess.STDOUT,
                                  stdin=subprocess.PIPE, text=True) for g in range(NGPU)]
        for p in procs:
            p.stdin.write("n\n"); p.stdin.close()
        # poll: stop all shards at once if one fails or the time budget runs out
        bad = []
        while any(p.poll() is None for p in procs):
            bad = [g for g, p in enumerate(procs) if p.poll() not in (None, 0)]
            late = hours_left() * 3600 < 600
            if bad or late:
                for p in procs:
                    if p.poll() is None:
                        p.kill()
                for p in procs:
                    p.wait()
                _clean_partial(t0)
                if late and not bad:
                    raise OutOfTime("time budget reached during collection: Save Version; attach this output and run again")
                break
            time.sleep(10)
        for f in logs:
            f.close()
        for g, p in sorted(enumerate(procs), key=lambda x: x[0] not in bad):   # the shard that failed first
            if p.returncode != 0:
                print(open(f"/root/shard{g}.log").read()[-3000:])
                raise RuntimeError(f"shard {g} failed")
    run(["scripts/01_collect.py"] + args, tail=tail)   # merge pass (or the single-GPU run)

In [ ]:
# 1f. Two GPUs in parallel: shard g runs on cuda:g (rendering stays on GPU 0); shard 1 starts a few minutes later so
#     the two 3B-parameter loads do not overlap in host RAM. Top-ups (--target_successes) run in the merge pass only.
R_REF = 40
exp["reference_episodes_per_task"] = R_REF; write_cfg(DTYPE)
STAGGER_S = 300
def collect(args, tail=1500):
    if hours_left() < 0.5:
        raise OutOfTime("time budget reached: Save Version now; attach this output and run the same PART again")
    args = list(args) + ([] if "--skip_existing" in args else ["--skip_existing"])
    shard_args, skip = [], 0
    for k, x in enumerate(args):                 # top-ups only in the single merge pass
        if skip: skip -= 1; continue
        if x in ("--target_successes", "--max_extra_rounds"): skip = 1; continue
        shard_args.append(x)
    env = dict(os.environ); env.update(MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false",
                                       PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    if NGPU > 1 and "--tasks" not in args:
        t0 = time.time(); procs, logs = [], []
        for g in range(NGPU):
            if g:
                time.sleep(STAGGER_S)
            logs.append(open(f"/root/shard{g}.log", "w"))
            e = dict(env); e["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in range(NGPU))
            p = subprocess.Popen([PY, "scripts/01_collect.py"] + shard_args + ["--device", f"cuda:{g}", "--tasks",
                                 ",".join(str(t) for t in range(10) if t % NGPU == g)],
                                 env=e, stdout=logs[g], stderr=subprocess.STDOUT, stdin=subprocess.PIPE, text=True)
            p.stdin.write("n\n"); p.stdin.close(); procs.append(p)
        bad = []
        while any(p.poll() is None for p in procs):
            bad = [g for g, p in enumerate(procs) if p.poll() not in (None, 0)]
            late = hours_left() * 3600 < 900
            if bad or late:
                for p in procs:
                    if p.poll() is None: p.kill()
                for p in procs: p.wait()
                _clean_partial(t0)
                if late and not bad:
                    raise OutOfTime("time budget reached during collection: Save Version; attach this output and rerun")
                break
            time.sleep(15)
        for f in logs: f.close()
        for g, p in enumerate(procs):
            if p.returncode != 0:
                print(open(f"/root/shard{g}.log").read()[-3000:])
                raise RuntimeError(f"shard {g} failed")
    e = dict(env); e["CUDA_VISIBLE_DEVICES"] = "0"
    p = subprocess.run([PY, "scripts/01_collect.py"] + args, capture_output=True, text=True, env=e, input="n\n")
    out = "\n".join(l for l in (p.stdout + p.stderr).replace("\r", "\n").splitlines() if "it/s" not in l)
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError("merge pass failed")
print("parallel collect ready on", NGPU, "GPU(s)")

## 2. Runs

In [ ]:
# 2a. Which part this commit runs
PART = 1
VARIANTS = {1: [], 2: ["w4", "w8"], 3: ["w4_lm", "update"], 4: ["cfg_denoise5", "cfg_steps5"]}[PART]
print("PART", PART, "| variants:", VARIANTS)

In [ ]:
# 2b. Condition: the monitor needs failures and the paired analysis needs successes -> target 55-88 % fp16 success.
#     pi0.5 succeeds on 100 % of nominal LIBERO-Spatial (pilot), so candidates are checked in order with 5 episodes
#     per task (outside the slot partition) and the first in range is used; otherwise the one closest to 72 %.
#     The choice is saved, so later parts reuse it.
cf = WORK / "pi05_condition.json"
if cf.exists():
    SUITE, CONDITION = json.loads(cf.read_text())["choice"]
else:
    assert PART == 1, "condition not found: attach the output of the previous part"
    tried = {}
    for suite, cond in (("libero_spatial", "shift_strong"), ("libero_spatial", "shift_mild"),
                        ("libero_10", "nominal"), ("libero_spatial", "action_noise")):
        out = WORK / f"cond_pilot/{suite}_{cond}"
        collect(["--model", MODEL, "--variant", "fp16", "--suite", suite, "--condition", cond, "--episodes", "5",
                 "--out", str(out)], tail=400)
        rj = json.loads(next(out.rglob("run.json")).read_text())
        tried[(suite, cond)] = rj["success_rate"]
        print(f"  {suite} + {cond}: fp16 success {rj['success_rate']:.2f} | {rj['seconds'] / rj['episodes']:.0f} s/episode")
        if 0.55 <= rj["success_rate"] <= 0.88:
            break
    SUITE, CONDITION = min(tried, key=lambda k: abs(tried[k] - 0.72))
    cf.write_text(json.dumps({"choice": [SUITE, CONDITION], "tried": {f"{s}+{c}": v for (s, c), v in tried.items()}}))
KEY = SUITE if CONDITION == "nominal" else f"{SUITE}+{CONDITION}"
print("Using", SUITE, "+", CONDITION, "| key", KEY)

In [ ]:
# 2c. Reference run (PART 1): 400 pre-registered slots (100 train / 80 calib / 80 repair / 140 test), resumable
ref = CODE / f"rollouts/{MODEL}/fp16/{KEY}"
if PART == 1 or len(list(ref.glob("*.npz"))) < 10 * R_REF:
    collect(["--model", MODEL, "--variant", "fp16", "--suite", SUITE, "--condition", CONDITION, "--episodes", str(R_REF)])
n_ref = len(list(ref.glob("*.npz")))
rj = json.loads((ref / "run.json").read_text())
print(f"reference episodes on disk: {n_ref} | fp16 success {rj['success_rate']:.3f}")
assert n_ref >= 10 * R_REF, "reference incomplete: attach the previous output and run PART 1 again"

In [ ]:
# 2d. Variants on the paired test slots + their own calibration pool (repair slots, top-ups to 44 successes)
for v in VARIANTS:
    print("=" * 30, v, flush=True)
    collect(["--model", MODEL, "--variant", v, "--suite", SUITE, "--condition", CONDITION, "--episodes", str(R_REF),
             "--roles", "test,repair", "--target_successes", "44", "--max_extra_rounds", "4"], tail=600)
    rj = json.loads((CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json").read_text())
    print(f"{v}: success {rj['success_rate']:.3f} | sampler {rj.get('sampler')}")

In [ ]:
# 2e. Score every variant run so far (CPU part): 5 monitor families x 3 placements + 20 re-splits + release options
import pandas as pd, numpy as np
done = sorted(p.name for p in (CODE / f"rollouts/{MODEL}").iterdir() if p.name != "fp16" and (p / KEY / "run.json").exists())
print("variants on disk:", done)
files = [cf, CODE / f"rollouts/{MODEL}/fp16/{KEY}/run.json"]
if done:
    for name, taps in (("both_layers", "vlm_last,expert_last"), ("expert_only", "expert_last"), ("vlm_only", "vlm_last")):
        run(["scripts/03_evaluate.py", "--model", MODEL, "--key", KEY,
             "--monitors", "mahalanobis,rnd,safe_mlp,acc,ext_mahalanobis", "--taps", taps], tail=400)
        dst = WORK / f"pi05_trials_{name}.csv"; shutil.copy(CODE / f"results/eval_trials/{MODEL}__{KEY}.csv", dst); files.append(dst)
        run(["scripts/resplit.py", "--model", MODEL, "--key", KEY, "--taps", taps, "--variants", ",".join(done),
             "--splits", "20", "--out", str(WORK / f"pi05_resplit_{name}.json")], tail=6000)
        files.append(WORK / f"pi05_resplit_{name}.json")
    for v in done:
        files.append(CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json")
with zipfile.ZipFile(f"/kaggle/working/pi05_part{PART}.zip", "w") as z:
    for p in files:
        p = pathlib.Path(p)
        if p.exists():
            z.write(p, p.name if "rollouts" not in str(p) else f"run_{p.parent.parent.name}.json")
print(f"Send me /kaggle/working/pi05_part{PART}.zip (Output tab)")